In [36]:
import pandas as pd
import numpy as np

timetable_by_course_code = pd.read_excel('Data/Timetable_by_course_code_2025_26.xlsx')
timetable_by_course_code.rename(columns={'Module.Host Key': 'Module'}, inplace=True)
timetable_by_course_code['Code'] = timetable_by_course_code['Description'].str.split(' ').str[0]
timetable_by_course_code['Start'] = pd.to_timedelta(timetable_by_course_code['Start'] + ':00')
timetable_by_course_code['End'] = pd.to_timedelta(timetable_by_course_code['End'] + ':00')
timetable_by_course_code['Duration'] = timetable_by_course_code['End'] - timetable_by_course_code['Start']
timetable_by_course_code['AT'] =  timetable_by_course_code['AT Weeks'].notna() & (timetable_by_course_code['AT Weeks'] != '')
timetable_by_course_code['WT'] =  timetable_by_course_code['WT Weeks'].notna() & (timetable_by_course_code['WT Weeks'] != '')
timetable_by_course_code['AT_week_count'] = timetable_by_course_code['AT Weeks'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) and x != '' else 0)
timetable_by_course_code['WT_week_count'] = timetable_by_course_code['WT Weeks'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) and x != '' else 0)
timetable_by_course_code.head()

,Department Name,Module,Description,Type,Group,Scheduled Day,Start,End,AT Weeks,WT Weeks,ST Weeks,Location,Staff,Code,Duration,AT,WT,AT_week_count,WT_week_count
0,AC00 - Department of Accounting,AC102,AC102 Elements of Financial Accounting,LECTURE,1,Mon,0 days 09:00:00,0 days 11:00:00,"1,2,3,4,5,6,7,8,9,10",NaN,NaN,PT,"KAMATH, DR SAIPRIYA",AC102,0 days 02:00:00,True,False,10,0
1,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,CLASS,1,Mon,0 days 11:00:00,0 days 12:00:00,"2,3,4,5,6,7,8,9,11",NaN,NaN,MAR.1.07,"CONSTANTINOU, MRS CHRISOULLA",AC102.A,0 days 01:00:00,True,False,9,0
2,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,CLASS,1,Thu,0 days 09:00:00,0 days 10:00:00,10,NaN,NaN,MAR.1.07,"CONSTANTINOU, MRS CHRISOULLA",AC102.A,0 days 01:00:00,True,False,1,0
3,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,CLASS,2,Mon,0 days 12:00:00,0 days 13:00:00,"2,3,4,5,6,7,8,9,11",NaN,NaN,MAR.1.07,"CONSTANTINOU, MRS CHRISOULLA",AC102.A,0 days 01:00:00,True,False,9,0
4,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,CLASS,2,Wed,0 days 09:00:00,0 days 10:00:00,10,NaN,NaN,MAR.1.09,"CONSTANTINOU, MRS CHRISOULLA",AC102.A,0 days 01:00:00,True,False,1,0


In [17]:
timetable_by_course_code['Type'].unique()

<StringArray>
['LECTURE', 'CLASS', 'SEMINAR', 'WORKSHOP']
Length: 4, dtype: str

In [18]:
teaching_rooms = pd.read_excel('Data/Teaching_rooms_2025_26.xlsx', header=2)
teaching_rooms.head()

,Zone Name,Name,Host Key,Capacity,Suitability
0,CBG - Centre Building,Auditorium [200],CBG.B1.02,200,T/Teaching Room
1,CBG - Centre Building,CBG.1.02 [42],CBG.1.02,42,T/Teaching Room
2,CBG - Centre Building,CBG.1.03 [42],CBG.1.03,42,T/Teaching Room
3,CBG - Centre Building,CBG.1.04 [42],CBG.1.04,42,T/Teaching Room
4,CBG - Centre Building,CBG.1.05 [42],CBG.1.05,42,T/Teaching Room


In [19]:
groups = pd.read_excel('Data/Group Sizes.xlsx')
groups.rename(columns={"Group No.": "Group"}, inplace=True)
groups.head()

,Code,Group,Teaching Title,Department,NUMSTU
0,AC102,L,Elements of Financial Accounting,Department of Accounting,646
1,AC102.A,1,Elements of Financial Accounting,Department of Accounting,19
2,AC102.A,10,Elements of Financial Accounting,Department of Accounting,18
3,AC102.A,11,Elements of Financial Accounting,Department of Accounting,16
4,AC102.A,12,Elements of Financial Accounting,Department of Accounting,18


In [20]:
groups['Group'].unique()

<StringArray>
[ 'L',  '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',  '2',
 '20', '21', '22', '23', '24', '25', '26', '27', '28', '29',  '3', '30', '31',
 '32', '33',  '4',  '5',  '6',  '7',  '8',  '9', '34', '35', '36', '37',  'W',
 '42', '43', '38', '39', '40', '41', '44']
Length: 46, dtype: str

In [47]:
course_master_data = timetable_by_course_code[['Code', 'Type', 'Group', 'Department Name', 'Module', 'Description', 'Duration', 'AT', 'AT Weeks', 'WT', 'WT Weeks', 'Staff']].copy()
course_master_data['Group'] = course_master_data['Group'].astype(str)
course_master_data.loc[course_master_data['Type'] == 'LECTURE', 'Group'] = 'L'
course_master_data.loc[course_master_data['Type'] == 'WORKSHOP', 'Group'] = 'W'
course_master_data = pd.merge(course_master_data, groups[['Code', 'Group', 'NUMSTU']].copy(), on=['Code', 'Group'])
course_master_data['Course_Level'] = course_master_data['Code'].str[2:3]
course_master_data.head()

,Code,Type,Group,Department Name,Module,Description,Duration,AT,AT Weeks,WT,WT Weeks,Staff,NUMSTU,Course_Level
0,AC102,LECTURE,L,AC00 - Department of Accounting,AC102,AC102 Elements of Financial Accounting,0 days 02:00:00,True,"1,2,3,4,5,6,7,8,9,10",False,NaN,"KAMATH, DR SAIPRIYA",646,1
1,AC102.A,CLASS,1,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,0 days 01:00:00,True,"2,3,4,5,6,7,8,9,11",False,NaN,"CONSTANTINOU, MRS CHRISOULLA",19,1
2,AC102.A,CLASS,1,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,0 days 01:00:00,True,10,False,NaN,"CONSTANTINOU, MRS CHRISOULLA",19,1
3,AC102.A,CLASS,2,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,0 days 01:00:00,True,"2,3,4,5,6,7,8,9,11",False,NaN,"CONSTANTINOU, MRS CHRISOULLA",18,1
4,AC102.A,CLASS,2,AC00 - Department of Accounting,AC102,AC102.A Elements of Financial Accounting,0 days 01:00:00,True,10,False,NaN,"CONSTANTINOU, MRS CHRISOULLA",18,1


In [52]:
course_master_data[course_master_data['Course_Level'].isin(['D', 'E'])]

,Code,Type,Group,Department Name,Module,Description,Duration,AT,AT Weeks,WT,WT Weeks,Staff,NUMSTU,Course_Level
5907,HYDIS,LECTURE,L,HY00 - International History,HYDIS,HYDIS HY300 Dissertation - year 2 meetings,0 days 01:00:00,False,NaN,True,3,"ALVANDI, DR ROHAM",59,D
5908,HYDIS,LECTURE,L,HY00 - International History,HYDIS,HYDIS HY300 Dissertation - year 2 meetings,0 days 01:00:00,False,NaN,True,4,"ALVANDI, DR ROHAM",59,D
7873,LSE100A,SEMINAR,1,LS00 - LSE100,LSE100A,LSE100A The LSE Course: How can we transform o...,0 days 01:30:00,True,"1,3,5,7,9",False,NaN,"TERRY, DR JILLIAN",28,E
7874,LSE100A,SEMINAR,1,LS00 - LSE100,LSE100A,LSE100A The LSE Course: How can we transform o...,0 days 01:30:00,False,NaN,True,"1,3,5,7,9","TERRY, DR JILLIAN",28,E
7875,LSE100A,SEMINAR,2,LS00 - LSE100,LSE100A,LSE100A The LSE Course: How can we transform o...,0 days 01:30:00,True,"1,3,5,7,9",False,NaN,"FRANKOVITCH, ALEX",23,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8031,LSE100C,SEMINAR,26,LS00 - LSE100,LSE100C,LSE100C The LSE Course: How can we create a fa...,0 days 01:30:00,False,NaN,True,"2,4,6,8,10","DAB, DR ALEXANDRE",25,E
8032,LSE100C,SEMINAR,27,LS00 - LSE100,LSE100C,LSE100C The LSE Course: How can we create a fa...,0 days 01:30:00,True,"2,4,6,8,10",False,NaN,"FRANKOVITCH, ALEX",25,E
8033,LSE100C,SEMINAR,27,LS00 - LSE100,LSE100C,LSE100C The LSE Course: How can we create a fa...,0 days 01:30:00,False,NaN,True,"2,4,6,8,10","FRANKOVITCH, ALEX",25,E
8034,LSE100C,SEMINAR,28,LS00 - LSE100,LSE100C,LSE100C The LSE Course: How can we create a fa...,0 days 01:30:00,True,"2,4,6,8,10",False,NaN,"DAB, DR ALEXANDRE",24,E


In [53]:
course_master_data[course_master_data['Department Name'] == 'MA00 - Mathematics']

,Code,Type,Group,Department Name,Module,Description,Duration,AT,AT Weeks,WT,WT Weeks,Staff,NUMSTU,Course_Level
8036,MA100.1,LECTURE,L,MA00 - Mathematics,MA100,MA100.1 Mathematical Methods - extra example s...,0 days 01:00:00,True,"1,2,3,5,6,7,8,9,10",False,NaN,"KOULETSIS, DR IOANNIS",518,1
8037,MA100,LECTURE,L,MA00 - Mathematics,MA100,MA100 Mathematical Methods,0 days 02:00:00,True,"1,2,3,5,6,7,8,9,10",False,NaN,"KOULETSIS, DR IOANNIS",518,1
8038,MA100.1,LECTURE,L,MA00 - Mathematics,MA100,MA100.1 Mathematical Methods - extra example s...,0 days 01:00:00,False,NaN,True,"1,2,3,4,5,6,7,8,9,10","KOULETSIS, DR IOANNIS",518,1
8039,MA100,LECTURE,L,MA00 - Mathematics,MA100,MA100 Mathematical Methods,0 days 02:00:00,False,NaN,True,"1,2,3","KOULETSIS, DR IOANNIS",518,1
8040,MA100,LECTURE,L,MA00 - Mathematics,MA100,MA100 Mathematical Methods,0 days 02:00:00,False,NaN,True,"4,5,6,7,8,9,10","KOULETSIS, DR IOANNIS",518,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8609,MA498,SEMINAR,1,MA00 - Mathematics,MA498,MA498 Dissertation in Mathematics,0 days 02:00:00,False,NaN,True,"1,2,3,4,5,6,7,8,9,10","ALLEN, PROF PETER",28,4
8610,MA498,SEMINAR,1,MA00 - Mathematics,MA498,MA498 Dissertation in Mathematics,0 days 02:00:00,False,NaN,False,NaN,"ALLEN, PROF PETER",28,4
8611,MA500,SEMINAR,1,MA00 - Mathematics,MA500,"MA500 Mathematics: Seminar on Combinatorics, G...",0 days 01:30:00,True,"1,2,3,4,5,6,7,8,9,10,11",False,NaN,"ASHKENAZI-GOLAN, DR GALIT_x000D_SAONA, DR RAIM...",5,5
8612,MA500,SEMINAR,1,MA00 - Mathematics,MA500,"MA500 Mathematics: Seminar on Combinatorics, G...",0 days 01:30:00,False,NaN,True,"1,2,3,4,5,6,7,8,9,10,11","ASHKENAZI-GOLAN, DR GALIT_x000D_SAONA, DR RAIM...",5,5


In [38]:
course_master_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 13431 entries, 0 to 13430
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype          
---  ------           --------------  -----          
 0   Code             13431 non-null  object         
 1   Type             13431 non-null  str            
 2   Group            13431 non-null  str            
 3   Department Name  13431 non-null  str            
 4   Module           13431 non-null  str            
 5   Description      13431 non-null  str            
 6   Duration         13431 non-null  timedelta64[us]
 7   AT               13431 non-null  bool           
 8   AT Weeks         5826 non-null   str            
 9   WT               13431 non-null  bool           
 10  WT Weeks         6337 non-null   str            
 11  Staff            13352 non-null  str            
dtypes: bool(2), object(1), str(8), timedelta64[us](1)
memory usage: 1.1+ MB


In [45]:
staff_df = course_master_data[['Staff', 'Code']].copy()

staff_df['First Name'] = staff_df['Staff'].str.split(',').str[0].str.strip()
staff_df['Title'] = staff_df['Staff'].str.split(',').str[1].str.strip().str.split().str[0]
staff_df['Last Name'] = staff_df['Staff'].str.split(',').str[1].str.strip().str.split().str[1:].str.join(' ')

staff_df = staff_df.dropna().drop_duplicates()

staff_df.head()

,Staff,Code,First Name,Title,Last Name
0,"KAMATH, DR SAIPRIYA",AC102,KAMATH,DR,SAIPRIYA
1,"CONSTANTINOU, MRS CHRISOULLA",AC102.A,CONSTANTINOU,MRS,CHRISOULLA
9,"IRGASHEV, MR KHAMID",AC102.A,IRGASHEV,MR,KHAMID
11,"MCDERMOTT, MR THOMAS",AC102.A,MCDERMOTT,MR,THOMAS
14,"MCDONALD, MR PHILIP",AC102.A,MCDONALD,MR,PHILIP
